In [17]:
from pathlib import Path

import pickle
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from config import (
    TAGS_EXCEL_PATH, DATA_CSV_PART1, TARGET_COL,
    FINAL_WEIGHTS, load_tag_lists,
)

## Данные

- `TAG_JOIN_IND` — техническая колонка. Её удаляем.
- Пропуски в TAG считаем нулями.
- Договоры без заполненных TAG не участвуют в расчёте.
- Ниже выводим число исключённых договоров.

In [18]:
tags_descriptions = pd.read_excel(TAGS_EXCEL_PATH, sheet_name='HT_list')
tag_lists = load_tag_lists(tags_descriptions)

data = pd.read_csv(DATA_CSV_PART1, encoding='cp1251', delimiter='^')
data.set_index('POLICY', inplace=True)

tag_cols = [col for col in data.columns if str(col).startswith("TAG_")]

data[tag_cols] = (
    data[tag_cols]
    .replace({r"\s+": "", ",": "."}, regex=True)
    .apply(pd.to_numeric,errors='coerce')
    .astype('float64')
)

data['SUM'] = data.filter(like='TAG_').fillna(0).sum(axis=1)
rows_without_tags = int(data['SUM'].le(0).sum())
data = data[data['SUM'] > 0].copy()

print('Исключено договоров без TAG:', rows_without_tags)
print('Осталось строк:', len(data))

Исключено договоров без TAG: 17622
Осталось строк: 114066


## Сырой скор

Для `auto_lover` и `shopping` умножаем каждый TAG на его вес и складываем. Для `alcohol` просто складываем значения TAG.

In [19]:
raw_scores = pd.DataFrame(index=data.index)
feature_tags = {}
weights = {}

group_settings = {
    'auto_lover': 'auto_lover_list',
    'shopping': 'shopping_features_list',
}

for group_name, tag_list_key in group_settings.items():
    selected_tags = list(tag_lists[tag_list_key])
    group_weights = {
        tag: float(FINAL_WEIGHTS[group_name].get(tag, 0.0))
        for tag in selected_tags
    }
    weights_series = pd.Series(group_weights, dtype=float)
    X = data.reindex(columns=selected_tags, fill_value=0).fillna(0).astype(float)
    raw_scores[f'{group_name}_raw_score'] = (
        X.mul(weights_series, axis=1).sum(axis=1)
    )
    feature_tags[group_name] = selected_tags
    weights[group_name] = group_weights

alcohol_tags = list(tag_lists['alcohol_features_list'])
X_alcohol = data.reindex(columns=alcohol_tags, fill_value=0).fillna(0).astype(float)
raw_scores['alcohol_raw_score'] = X_alcohol.sum(axis=1)
feature_tags['alcohol'] = alcohol_tags

raw_scores.head()

,auto_lover_raw_score,shopping_raw_score,alcohol_raw_score
POLICY,,,
077LL0600001924,0.0000,0.00,1.32000
077LL0600001934,6.5450,7.44,3.16761
077LL0600001939,10.2058,21.58,3.78209
077LL0600001955,0.0000,0.00,5.71176
077LL0600002002,4.6333,17.70,0.00000


## MinMaxScaler

Для каждой группы создаём отдельный MinMaxScaler. Здесь он настраивается на исходном датасете через `fit_transform`. Параметр `clip=True` ограничит будущие значения диапазоном от 0 до 1.

In [20]:
result_fit = pd.DataFrame(index=data.index)
scalers = {}

for group_name in ('auto_lover', 'shopping', 'alcohol'):
    raw_column = f'{group_name}_raw_score'
    scaler = MinMaxScaler(clip=True)
    result_fit[f'{group_name}_agg_coef'] = scaler.fit_transform(
        raw_scores[[raw_column]]
    ).ravel()
    scalers[group_name] = scaler

result_fit.head()

,auto_lover_agg_coef,shopping_agg_coef,alcohol_agg_coef
POLICY,,,
077LL0600001924,0.000000,0.000000,0.130634
077LL0600001934,0.190958,0.146226,0.313483
077LL0600001939,0.297766,0.424135,0.374295
077LL0600001955,0.000000,0.000000,0.565264
077LL0600002002,0.135182,0.347877,0.000000


## Сохранение

Сохраняем списки TAG, веса и обученные MinMaxScaler в `aggregated_tags_pipeline.pkl`. Рассчитанные признаки сохраняем в CSV.

In [21]:
artifact = {
    'version': '1.0',
    'feature_tags': feature_tags,
    'weights': weights,
    'scalers': scalers,
}

MODEL_DIR = Path('artifacts') / 'model'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'aggregated_tags_pipeline.pkl'
with MODEL_PATH.open('wb') as file:
    pickle.dump(artifact, file, protocol=pickle.HIGHEST_PROTOCOL)

CSV_DIR = Path('artifacts') / 'csv'
CSV_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = CSV_DIR / 'all_groups_agg_coef.csv'
result_fit.to_csv(CSV_PATH, index=True, encoding='utf-8-sig')

print('Сохранён pkl:', MODEL_PATH)
print('Сохранён CSV:', CSV_PATH)

Сохранён pkl: artifacts\model\aggregated_tags_pipeline.pkl
Сохранён CSV: artifacts\csv\all_groups_agg_coef.csv
